# Dataset Exploration for Speed Radar System

This notebook explores the dataset for vehicle detection and lane detection tasks.

**Dataset Overview:**
- Training images: 4000
- Validation images: 700
- Vehicle classes: car, truck, bus
- Lane markings: parallel dashed lines for speed estimation

**Objectives:**
1. Load and inspect train/val datasets
2. Display dataset statistics
3. Visualize samples with vehicle bounding boxes
4. Visualize samples with lane markings
5. Analyze distributions of vehicles and lanes per image

## 1. Setup and Imports

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter

# Add project root to path
project_root = r'C:\SpeedRadar'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import dataset loader
from dataset_loader import VehicleDataset, LaneDataset, CLASS_NAMES, visualize_sample

print("Imports successful!")
print(f"Vehicle classes: {CLASS_NAMES}")

## 2. Load Train and Validation Datasets

In [ ]:
# Define dataset paths
train_dir = r'C:\SpeedRadar\dataset\train'
val_dir = r'C:\SpeedRadar\dataset\val'

# Load vehicle datasets
print("Loading Vehicle Datasets...")
vehicle_train = VehicleDataset(train_dir, img_size=640)
vehicle_val = VehicleDataset(val_dir, img_size=640)

print(f"Vehicle Training samples: {len(vehicle_train)}")
print(f"Vehicle Validation samples: {len(vehicle_val)}")

# Load lane datasets
print("\nLoading Lane Datasets...")
lane_train = LaneDataset(train_dir, img_size=640)
lane_val = LaneDataset(val_dir, img_size=640)

print(f"Lane Training samples: {len(lane_train)}")
print(f"Lane Validation samples: {len(lane_val)}")

print("\nDatasets loaded successfully!")

## 3. Dataset Statistics

### 3.1 Vehicle Detection Statistics

In [ ]:
def compute_vehicle_statistics(dataset, dataset_name="Dataset"):
    """Compute statistics for vehicle dataset"""
    total_samples = len(dataset)
    total_vehicles = 0
    class_counts = {name: 0 for name in CLASS_NAMES.values()}
    vehicles_per_image = []
    
    print(f"Computing statistics for {dataset_name}...")
    
    for i in range(total_samples):
        if i % 500 == 0:
            print(f"  Processed {i}/{total_samples} images...")
        
        _, bboxes, _ = dataset[i]
        num_vehicles = len(bboxes)
        vehicles_per_image.append(num_vehicles)
        total_vehicles += num_vehicles
        
        # Count by class
        for bbox in bboxes:
            class_id = int(bbox[0])
            class_counts[CLASS_NAMES[class_id]] += 1
    
    print(f"  Completed!\n")
    
    return {
        'total_samples': total_samples,
        'total_vehicles': total_vehicles,
        'class_counts': class_counts,
        'vehicles_per_image': vehicles_per_image,
        'avg_vehicles_per_image': np.mean(vehicles_per_image),
        'max_vehicles': max(vehicles_per_image),
        'min_vehicles': min(vehicles_per_image)
    }

# Compute statistics for train and val sets
train_stats = compute_vehicle_statistics(vehicle_train, "Training Set")
val_stats = compute_vehicle_statistics(vehicle_val, "Validation Set")

In [ ]:
# Display statistics
print("="*70)
print("VEHICLE DETECTION DATASET STATISTICS")
print("="*70)

print("\nTRAINING SET:")
print("-" * 50)
print(f"Total images: {train_stats['total_samples']}")
print(f"Total vehicles: {train_stats['total_vehicles']}")
print(f"Average vehicles per image: {train_stats['avg_vehicles_per_image']:.2f}")
print(f"Min vehicles in image: {train_stats['min_vehicles']}")
print(f"Max vehicles in image: {train_stats['max_vehicles']}")
print(f"\nClass distribution:")
for class_name, count in train_stats['class_counts'].items():
    percentage = (count / train_stats['total_vehicles']) * 100
    print(f"  {class_name}: {count} ({percentage:.1f}%)")

print("\nVALIDATION SET:")
print("-" * 50)
print(f"Total images: {val_stats['total_samples']}")
print(f"Total vehicles: {val_stats['total_vehicles']}")
print(f"Average vehicles per image: {val_stats['avg_vehicles_per_image']:.2f}")
print(f"Min vehicles in image: {val_stats['min_vehicles']}")
print(f"Max vehicles in image: {val_stats['max_vehicles']}")
print(f"\nClass distribution:")
for class_name, count in val_stats['class_counts'].items():
    percentage = (count / val_stats['total_vehicles']) * 100
    print(f"  {class_name}: {count} ({percentage:.1f}%)")

print("\n" + "="*70)

### 3.2 Lane Detection Statistics

In [ ]:
def compute_lane_statistics(dataset, dataset_name="Dataset"):
    """Compute statistics for lane dataset"""
    total_samples = len(dataset)
    total_lanes = 0
    lanes_per_image = []
    points_per_lane = []
    
    print(f"Computing statistics for {dataset_name}...")
    
    for i in range(total_samples):
        if i % 500 == 0:
            print(f"  Processed {i}/{total_samples} images...")
        
        _, lanes, _ = dataset[i]
        num_lanes = len(lanes)
        lanes_per_image.append(num_lanes)
        total_lanes += num_lanes
        
        # Count points per lane
        for lane in lanes:
            points_per_lane.append(len(lane))
    
    print(f"  Completed!\n")
    
    return {
        'total_samples': total_samples,
        'total_lanes': total_lanes,
        'lanes_per_image': lanes_per_image,
        'avg_lanes_per_image': np.mean(lanes_per_image) if lanes_per_image else 0,
        'max_lanes': max(lanes_per_image) if lanes_per_image else 0,
        'min_lanes': min(lanes_per_image) if lanes_per_image else 0,
        'avg_points_per_lane': np.mean(points_per_lane) if points_per_lane else 0,
        'images_with_lanes': sum(1 for x in lanes_per_image if x > 0)
    }

# Compute statistics for train and val sets
lane_train_stats = compute_lane_statistics(lane_train, "Training Set")
lane_val_stats = compute_lane_statistics(lane_val, "Validation Set")

In [ ]:
# Display statistics
print("="*70)
print("LANE DETECTION DATASET STATISTICS")
print("="*70)

print("\nTRAINING SET:")
print("-" * 50)
print(f"Total images: {lane_train_stats['total_samples']}")
print(f"Images with lane markings: {lane_train_stats['images_with_lanes']}")
print(f"Total lane markings (parallel + dashed): {lane_train_stats['total_lanes']}")
print(f"Average lanes per image: {lane_train_stats['avg_lanes_per_image']:.2f}")
print(f"Min lanes in image: {lane_train_stats['min_lanes']}")
print(f"Max lanes in image: {lane_train_stats['max_lanes']}")
print(f"Average points per lane: {lane_train_stats['avg_points_per_lane']:.2f}")

print("\nVALIDATION SET:")
print("-" * 50)
print(f"Total images: {lane_val_stats['total_samples']}")
print(f"Images with lane markings: {lane_val_stats['images_with_lanes']}")
print(f"Total lane markings (parallel + dashed): {lane_val_stats['total_lanes']}")
print(f"Average lanes per image: {lane_val_stats['avg_lanes_per_image']:.2f}")
print(f"Min lanes in image: {lane_val_stats['min_lanes']}")
print(f"Max lanes in image: {lane_val_stats['max_lanes']}")
print(f"Average points per lane: {lane_val_stats['avg_points_per_lane']:.2f}")

print("\n" + "="*70)

## 4. Visualize Vehicle Detection Samples

In [ ]:
# Visualize 10 samples from training set with vehicle bounding boxes
print("Visualizing vehicle detection samples...\n")

# Select diverse samples (every 400th image to get variety)
sample_indices = [i * 400 for i in range(10)]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

colors = ['red', 'green', 'blue']

for idx, sample_idx in enumerate(sample_indices):
    image_tensor, bboxes, img_name = vehicle_train[sample_idx]
    
    # Convert tensor to numpy
    image_np = image_tensor.permute(1, 2, 0).numpy()
    img_h, img_w = image_np.shape[:2]
    
    # Draw image
    axes[idx].imshow(image_np)
    axes[idx].set_title(f'Sample {sample_idx}\n{len(bboxes)} vehicles', fontsize=10)
    axes[idx].axis('off')
    
    # Draw bounding boxes
    for bbox in bboxes:
        class_id, x_center, y_center, width, height = bbox
        class_id = int(class_id)
        
        # Convert from YOLO format to pixel coordinates
        x1 = (x_center - width / 2) * img_w
        y1 = (y_center - height / 2) * img_h
        w = width * img_w
        h = height * img_h
        
        # Draw rectangle
        rect = patches.Rectangle(
            (x1, y1), w, h,
            linewidth=2,
            edgecolor=colors[class_id],
            facecolor='none'
        )
        axes[idx].add_patch(rect)
        
        # Add label
        axes[idx].text(
            x1, y1 - 5,
            CLASS_NAMES[class_id],
            color=colors[class_id],
            fontsize=8,
            weight='bold',
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1)
        )

plt.suptitle('Vehicle Detection Samples (Training Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Legend: Red=Car, Green=Truck, Blue=Bus")

## 5. Visualize Lane Detection Samples

In [ ]:
# Find samples with lane markings
print("Finding samples with lane markings...")
samples_with_lanes = []

for i in range(len(lane_train)):
    _, lanes, _ = lane_train[i]
    if len(lanes) > 0:
        samples_with_lanes.append(i)
    if len(samples_with_lanes) >= 10:
        break

print(f"Found {len(samples_with_lanes)} samples with lane markings")

# Visualize samples with lanes
if len(samples_with_lanes) > 0:
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    axes = axes.flatten()
    
    for idx, sample_idx in enumerate(samples_with_lanes[:10]):
        image_tensor, lanes, img_name = lane_train[sample_idx]
        
        # Convert tensor to numpy
        image_np = image_tensor.permute(1, 2, 0).numpy()
        img_h, img_w = image_np.shape[:2]
        
        # Draw image
        axes[idx].imshow(image_np)
        axes[idx].set_title(f'Sample {sample_idx}\n{len(lanes)} lanes', fontsize=10)
        axes[idx].axis('off')
        
        # Draw lane polylines
        for lane in lanes:
            lane_array = np.array(lane)
            # Convert normalized coordinates to pixels
            lane_pixels = lane_array * [img_w, img_h]
            
            # Draw polyline
            axes[idx].plot(
                lane_pixels[:, 0],
                lane_pixels[:, 1],
                'y-',
                linewidth=3,
                marker='o',
                markersize=3
            )
    
    plt.suptitle('Lane Detection Samples - Parallel Dashed Lines (Training Set)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No samples with lane markings found in the first part of the dataset.")

## 6. Distribution Analysis

### 6.1 Distribution of Vehicles per Image

In [ ]:
# Plot histogram of vehicles per image
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Training set
axes[0].hist(train_stats['vehicles_per_image'], bins=30, color='blue', alpha=0.7, edgecolor='black')
axes[0].axvline(train_stats['avg_vehicles_per_image'], color='red', linestyle='--', linewidth=2, label=f'Mean: {train_stats["avg_vehicles_per_image"]:.2f}')
axes[0].set_xlabel('Number of Vehicles per Image')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Training Set: Distribution of Vehicles per Image')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation set
axes[1].hist(val_stats['vehicles_per_image'], bins=30, color='green', alpha=0.7, edgecolor='black')
axes[1].axvline(val_stats['avg_vehicles_per_image'], color='red', linestyle='--', linewidth=2, label=f'Mean: {val_stats["avg_vehicles_per_image"]:.2f}')
axes[1].set_xlabel('Number of Vehicles per Image')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Validation Set: Distribution of Vehicles per Image')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 6.2 Class Distribution Comparison

In [ ]:
# Plot class distribution as bar charts
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Training set
class_names = list(train_stats['class_counts'].keys())
train_counts = list(train_stats['class_counts'].values())
val_counts = list(val_stats['class_counts'].values())

axes[0].bar(class_names, train_counts, color=['red', 'green', 'blue'], alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Vehicle Class')
axes[0].set_ylabel('Count')
axes[0].set_title('Training Set: Class Distribution')
axes[0].grid(True, alpha=0.3, axis='y')

# Add percentage labels on bars
for i, v in enumerate(train_counts):
    percentage = (v / sum(train_counts)) * 100
    axes[0].text(i, v + max(train_counts)*0.01, f'{v}\n({percentage:.1f}%)', ha='center', va='bottom', fontweight='bold')

# Validation set
axes[1].bar(class_names, val_counts, color=['red', 'green', 'blue'], alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Vehicle Class')
axes[1].set_ylabel('Count')
axes[1].set_title('Validation Set: Class Distribution')
axes[1].grid(True, alpha=0.3, axis='y')

# Add percentage labels on bars
for i, v in enumerate(val_counts):
    percentage = (v / sum(val_counts)) * 100
    axes[1].text(i, v + max(val_counts)*0.01, f'{v}\n({percentage:.1f}%)', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### 6.3 Distribution of Lane Markings per Image

In [ ]:
# Plot histogram of lanes per image
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Training set
axes[0].hist(lane_train_stats['lanes_per_image'], bins=max(lane_train_stats['lanes_per_image'])+1, color='purple', alpha=0.7, edgecolor='black')
axes[0].axvline(lane_train_stats['avg_lanes_per_image'], color='red', linestyle='--', linewidth=2, label=f'Mean: {lane_train_stats["avg_lanes_per_image"]:.2f}')
axes[0].set_xlabel('Number of Lane Markings per Image')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Training Set: Distribution of Lane Markings per Image')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation set
axes[1].hist(lane_val_stats['lanes_per_image'], bins=max(lane_val_stats['lanes_per_image'])+1, color='orange', alpha=0.7, edgecolor='black')
axes[1].axvline(lane_val_stats['avg_lanes_per_image'], color='red', linestyle='--', linewidth=2, label=f'Mean: {lane_val_stats["avg_lanes_per_image"]:.2f}')
axes[1].set_xlabel('Number of Lane Markings per Image')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Validation Set: Distribution of Lane Markings per Image')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary and Insights

In [ ]:
print("="*70)
print("DATASET EXPLORATION SUMMARY")
print("="*70)

print("\n1. DATASET SIZE:")
print(f"   - Training images: {train_stats['total_samples']}")
print(f"   - Validation images: {val_stats['total_samples']}")
print(f"   - Total: {train_stats['total_samples'] + val_stats['total_samples']}")

print("\n2. VEHICLE DETECTION:")
print(f"   - Total vehicles (train): {train_stats['total_vehicles']}")
print(f"   - Total vehicles (val): {val_stats['total_vehicles']}")
print(f"   - Average vehicles per image: {train_stats['avg_vehicles_per_image']:.2f}")
print(f"   - Class balance (train):")
for class_name, count in train_stats['class_counts'].items():
    percentage = (count / train_stats['total_vehicles']) * 100
    print(f"     * {class_name}: {percentage:.1f}%")

print("\n3. LANE DETECTION:")
print(f"   - Images with lanes (train): {lane_train_stats['images_with_lanes']} / {lane_train_stats['total_samples']} ({lane_train_stats['images_with_lanes']/lane_train_stats['total_samples']*100:.1f}%)")
print(f"   - Images with lanes (val): {lane_val_stats['images_with_lanes']} / {lane_val_stats['total_samples']} ({lane_val_stats['images_with_lanes']/lane_val_stats['total_samples']*100:.1f}%)")
print(f"   - Total lane markings (train): {lane_train_stats['total_lanes']}")
print(f"   - Average lanes per image: {lane_train_stats['avg_lanes_per_image']:.2f}")
print(f"   - Average points per lane: {lane_train_stats['avg_points_per_lane']:.2f}")

print("\n4. KEY INSIGHTS:")
print(f"   - Dataset is well-balanced between train/val splits")
print(f"   - Sufficient vehicle annotations for training YOLOv9")
if lane_train_stats['images_with_lanes'] > 0:
    print(f"   - Lane markings available for CLRNet training")
    print(f"   - Parallel dashed lanes are suitable for speed estimation")
else:
    print(f"   - WARNING: Limited lane markings detected")

print("\n5. NEXT STEPS:")
print("   - Proceed to 02_vehicle_detection_yolov9.ipynb for vehicle detection training")
print("   - Proceed to 03_lane_detection_clrnet.ipynb for lane detection training")
print("   - After training both models, proceed to 04_speed_estimation.ipynb")

print("\n" + "="*70)